In [1]:
import pandas as pd
from pathlib import Path
import json

In [2]:
path = Path(r"../results/sample_500_bluesky_llama_results.csv")
# Example alternate paths:
# path = Path(r"../important_results/full_bluesky_gpt_results.csv")
# path = Path(r"../important_results/sample_500_meta_gpt_results.csv")
df = pd.read_csv(
    path,
    encoding="latin-1",
    on_bad_lines="skip"
)

In [3]:
if "ad_text" in df.columns:
    text_col = "ad_text"
elif "text" in df.columns:
    text_col = "text"
else:
    raise ValueError(f"No text column found. Available columns: {list(df.columns)}")

if "gpt_toulmin_json" not in df.columns and "pred_toulmin_json" not in df.columns:
    raise ValueError("Column 'gpt_toulmin_json' not found.")

def extract_claim(toulmin_json, key = "claim"):
    if pd.isna(toulmin_json):
        return "Not present"
    s = str(toulmin_json).strip()
    if not s:
        return "Not present"
    try:
        obj = json.loads(s)
        if isinstance(obj, dict):
            claim = obj.get(key, "Not present")
            claim = "Not present" if claim is None else str(claim).strip()
            return claim if claim else "Not present"
    except Exception:
        pass
    return "Not present"

def build_input_text(text, toulmin_json, key = "claim"):
    text = "" if pd.isna(text) else str(text)
    claim = extract_claim(toulmin_json, key)
    return f"Text:\n{text}\nOutput:\n{claim}"

In [4]:
df["input_text"] = [
    build_input_text(text, tj, "ground")
    for text, tj in zip(df[text_col], df["pred_toulmin_json"])
]

output_path = path.with_name(path.stem + "_with_ground_input_text.csv")
df.to_csv(output_path, index=False)
print(f"Saved: {output_path}")

Saved: ..\results\sample_500_bluesky_llama_results_with_ground_input_text.csv


In [12]:
df["claim"] = [
    extract_claim(toulmin_json)
    for toulmin_json in df["pred_toulmin_json"]
]

output_path = path.with_name(path.stem + "_with_claims.csv")
df.to_csv(output_path, index=False)
print(f"Saved: {output_path}")

Saved: ..\important_results\full_bluesky_llama_results_merged_with_claims.csv


In [13]:
df["ground"] = [
    extract_claim(toulmin_json, key="ground")
    for toulmin_json in df["pred_toulmin_json"]
]

output_path = path.with_name(path.stem + "_with_grounds.csv")
df.to_csv(output_path, index=False)
print(f"Saved: {output_path}")

Saved: ..\important_results\full_bluesky_llama_results_merged_with_grounds.csv


In [12]:
path = Path(r"../results/sample_500_bluesky_llama_results.csv")
# Example alternate paths:
# path = Path(r"../important_results/full_bluesky_gpt_results.csv")
# path = Path(r"../important_results/sample_500_meta_gpt_results.csv")
df = pd.read_csv(path)

In [13]:
def is_valid_json(s):
    if pd.isna(s) or str(s).strip() in {"", "None", "null"}:
        return False
    try:
        return json.loads(s) is not None
    except Exception:
        return False

df = df[df["pred_toulmin_json"].apply(is_valid_json)]

In [14]:
if "ad_text" in df.columns:
    text_col = "ad_text"
elif "text" in df.columns:
    text_col = "text"
else:
    raise ValueError(f"No text column found. Available columns: {list(df.columns)}")

if "gpt_toulmin_json" not in df.columns and "pred_toulmin_json" not in df.columns:
    raise ValueError("Column 'gpt_toulmin_json' not found.")

def extract_claim_ground(toulmin_json):
    if pd.isna(toulmin_json):
        return "Not present"
    s = str(toulmin_json).strip()
    if not s:
        return "Not present"
    try:
        obj = json.loads(s)
        if isinstance(obj, dict):
            claim = obj.get("claim", "Not present")
            claim = "Not present" if claim is None else str(claim).strip()

            ground = obj.get("ground", "Not present")
            ground = "Not present" if ground is None else str(ground).strip()
            return claim, ground

    except Exception as e:
        print(e)
    return "Not present", "Not present"

def build_ground_input_text(text, toulmin_json):
    text = "" if pd.isna(text) else str(text)
    claim, ground = extract_claim_ground(toulmin_json)
    return f"Text:\n{text}\nClaim:\n{claim}\nGround:\n{ground}"

In [15]:
df["ground_input_text"] = [
    build_ground_input_text(text, tj)
    for text, tj in zip(df[text_col], df["pred_toulmin_json"])
]

output_path = path.with_name(path.stem + "_with_ground_input_text.csv")
df.to_csv(output_path, index=False)
print(f"Saved: {output_path}")

Saved: ..\results\sample_500_bluesky_llama_results_with_ground_input_text.csv


In [ ]:
df[pd.isna(df['pred_toulmin_json'])]

ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

In [20]:
n = 10

for res in df["ground_input_text"][:n]:
    print(res)

Text:
🛑🚨🛑🚨Protecting the Endangerment Finding = Protecting Children's Health! Moms across the country share what rolling back this foundational protection for regulating climate emissions means to them. Listen in and then tell EPA to do its job and keep cutting climate pollution!
Claim:
Protecting the Endangerment Finding protects children's health.
Ground:
Rolling back this foundational protection for regulating climate emissions means to moms across the country.
Text:
Stop Kamala's Radical Plan for America Before It's Too Late

Kamala Harris is dangerously liberal, and her extreme policies threaten to fundamentally weaken our nation. Under her leadership, America could face devastating consequences: higher taxes, lenient crime policies, and an alarming erosion of our freedoms. Harris's radical agenda includes fighting to set murderers free, embracing the disastrous "Bidenomics" that dismisses rising costs, and imposing more taxes on the middle class under the guise of "fair share." S